# Thực nghiệm DNA Matching trên full genome NCBI

Notebook này chạy trực tiếp trên sequence NCBI dài `4,641,652` ký tự trong file `GCF_000005845.2_ASM584v2_genomic.fna`, không chia nhỏ text thành các lát cắt.

Các thuật toán được so sánh theo danh sách trong `main.py`:

- Brute Force with k mismatches
- Rabin-Karp with Verification
- KMP-based Filtering + Verification
- Boyer-Moore with Verification
- Suffix Array with Candidate Verification
- Suffix Tree with Backtracking

Lưu ý quan trọng: code hiện tại của `suffix_array` xây suffix array bằng `sorted(range(len(text)), key=lambda i: text[i:])`, còn `suffix_tree` xây cây trực tiếp trong Python. Với full genome hơn 4.6 triệu ký tự, hai thuật toán này có thể chạy rất lâu hoặc hết RAM. Notebook vẫn thử chạy chúng, nhưng có timeout để không làm treo toàn bộ thí nghiệm.

In [ ]:
from pathlib import Path
import multiprocessing as mp
import os
import sys
import time
import traceback

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-dna-matching")

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "experiments":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from main import get_algorithm, read_sequence

NCBI_FASTA = PROJECT_ROOT / "dataset" / "ncbi" / "data" / "GCF_000005845.2" / "GCF_000005845.2_ASM584v2_genomic.fna"
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "ncbi_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CANONICAL_ALGORITHMS = [
    "brute_force",
    "rabin_karp",
    "kmp",
    "boyer_moore",
    "suffix_array",
    "suffix_tree",
]

FULL_RUN_TIMEOUT_SEC = 180
STOP_ALGORITHM_AFTER_TIMEOUT = True

print("Project root:", PROJECT_ROOT)
print("NCBI FASTA:", NCBI_FASTA)
print("Output dir:", OUTPUT_DIR)
print("Per-run timeout:", FULL_RUN_TIMEOUT_SEC, "seconds")

## 1. Nạp full genome NCBI

In [ ]:
raw_sequence = read_sequence(NCBI_FASTA)
full_sequence = "".join(ch for ch in raw_sequence if ch in "ACGT")
GENOME_NAME = NCBI_FASTA.parent.name

sequence_summary = pd.DataFrame([{
    "genome": GENOME_NAME,
    "length_bp": len(full_sequence),
    "source_file": str(NCBI_FASTA.relative_to(PROJECT_ROOT)),
}])
sequence_summary

## 2. Cấu hình case thực nghiệm

`text` của mọi case là toàn bộ genome dài `4,641,652` ký tự. Các case chỉ thay đổi `pattern_length` và `k`. Pattern được lấy từ chính full genome tại vị trí khoảng 1/3 chiều dài. Với `k > 0`, pattern được mutate đúng `k` vị trí để vẫn có nghiệm approximate tại `source_start`.

In [ ]:
PATTERN_LENGTHS = [12, 24, 48]
K_VALUES = [0, 1, 2]
BASES = "ACGT"

def mutate_pattern(pattern: str, mismatches: int) -> str:
    if mismatches <= 0:
        return pattern
    chars = list(pattern)
    positions = sorted({round(i * (len(chars) - 1) / max(1, mismatches - 1)) for i in range(mismatches)})
    for pos in positions[:mismatches]:
        current = chars[pos]
        choices = [base for base in BASES if base != current]
        chars[pos] = choices[(pos + mismatches) % len(choices)]
    return "".join(chars)

def make_full_genome_cases(sequence: str) -> list[dict]:
    cases = []
    text_length = len(sequence)
    for pattern_length in PATTERN_LENGTHS:
        source_start = max(0, min(text_length - pattern_length, text_length // 3))
        source_pattern = sequence[source_start:source_start + pattern_length]
        for k in K_VALUES:
            if k >= pattern_length:
                continue
            cases.append({
                "genome": GENOME_NAME,
                "text_length": text_length,
                "pattern_length": pattern_length,
                "k": k,
                "source_start": source_start,
                "text": sequence,
                "pattern": mutate_pattern(source_pattern, k),
            })
    return cases

cases = make_full_genome_cases(full_sequence)
case_summary = pd.DataFrame([{k: v for k, v in case.items() if k not in {"text", "pattern"}} for case in cases])
print("Number of cases:", len(cases))
case_summary

## 3. Chạy toàn bộ thuật toán trên full genome

Mỗi lượt chạy được bọc trong process riêng để áp timeout. Nếu một thuật toán timeout và `STOP_ALGORITHM_AFTER_TIMEOUT = True`, các case sau của thuật toán đó được đánh dấu `skipped_after_timeout` để tiết kiệm thời gian.

In [ ]:
def _algorithm_worker(queue, algorithm_key: str, text: str, pattern: str, k: int):
    try:
        algorithm = get_algorithm(algorithm_key)
        start = time.perf_counter()
        matches = algorithm.run(text, pattern, k)
        runtime = time.perf_counter() - start
        queue.put({
            "status": "ok",
            "algorithm": algorithm.name,
            "runtime_sec": runtime,
            "num_matches": len(matches),
            "matches_preview": matches[:10],
            "source_matches": matches,
        })
    except Exception as exc:
        queue.put({
            "status": "error",
            "algorithm": algorithm_key,
            "runtime_sec": None,
            "num_matches": None,
            "matches_preview": [],
            "source_matches": [],
            "error": f"{type(exc).__name__}: {exc}",
            "traceback": traceback.format_exc(limit=5),
        })

def run_with_timeout(algorithm_key: str, text: str, pattern: str, k: int, timeout_sec: int) -> dict:
    queue = mp.Queue(maxsize=1)
    process = mp.Process(target=_algorithm_worker, args=(queue, algorithm_key, text, pattern, k))
    start = time.perf_counter()
    process.start()
    process.join(timeout_sec)
    elapsed = time.perf_counter() - start

    if process.is_alive():
        process.terminate()
        process.join(5)
        return {
            "status": "timeout",
            "algorithm": algorithm_key,
            "runtime_sec": elapsed,
            "num_matches": None,
            "matches_preview": [],
            "source_matches": [],
            "error": f"Timeout after {timeout_sec} seconds",
        }

    if queue.empty():
        return {
            "status": "error",
            "algorithm": algorithm_key,
            "runtime_sec": elapsed,
            "num_matches": None,
            "matches_preview": [],
            "source_matches": [],
            "error": f"Process exited with code {process.exitcode} and returned no result",
        }

    return queue.get()

rows = []
timed_out_algorithms = set()
total_runs = len(cases) * len(CANONICAL_ALGORITHMS)
run_index = 0

for algorithm_key in CANONICAL_ALGORITHMS:
    for case in cases:
        run_index += 1
        print(
            f"[{run_index}/{total_runs}] {algorithm_key} | "
            f"n={case['text_length']} | m={case['pattern_length']} | k={case['k']}",
            flush=True,
        )

        if STOP_ALGORITHM_AFTER_TIMEOUT and algorithm_key in timed_out_algorithms:
            result = {
                "status": "skipped_after_timeout",
                "algorithm": algorithm_key,
                "runtime_sec": None,
                "num_matches": None,
                "matches_preview": [],
                "source_matches": [],
                "error": "Skipped because a previous full-genome case for this algorithm timed out",
            }
        else:
            result = run_with_timeout(algorithm_key, case["text"], case["pattern"], case["k"], FULL_RUN_TIMEOUT_SEC)
            if result["status"] == "timeout":
                timed_out_algorithms.add(algorithm_key)

        source_found = case["source_start"] in result.get("source_matches", [])
        rows.append({
            "genome": case["genome"],
            "text_length": case["text_length"],
            "pattern_length": case["pattern_length"],
            "k": case["k"],
            "source_start": case["source_start"],
            "source_found": source_found if result["status"] == "ok" else None,
            "algorithm_key": algorithm_key,
            "algorithm": result["algorithm"],
            "status": result["status"],
            "runtime_sec": result["runtime_sec"],
            "num_matches": result["num_matches"],
            "matches_preview": result["matches_preview"],
            "error": result.get("error"),
        })

results = pd.DataFrame(rows)
results_path = OUTPUT_DIR / "ncbi_full_genome_experiment_results.csv"
results.to_csv(results_path, index=False)
print("Saved:", results_path)
results

In [ ]:
status_summary = (
    results.groupby(["algorithm_key", "status"], dropna=False, as_index=False)
    .size()
    .rename(columns={"size": "runs"})
)
status_summary

In [ ]:
accuracy_summary = (
    results[results["status"] == "ok"]
    .groupby(["algorithm_key", "k"], as_index=False)
    .agg(total_ok_runs=("source_found", "size"), source_found_runs=("source_found", "sum"))
)
if not accuracy_summary.empty:
    accuracy_summary["source_found_rate"] = accuracy_summary["source_found_runs"] / accuracy_summary["total_ok_runs"]
accuracy_summary

## 4. Tổng hợp kết quả

In [ ]:
completed = results[results["status"] == "ok"].copy()
summary = (
    completed.groupby(["algorithm_key", "algorithm", "text_length", "pattern_length", "k"], as_index=False)
    .agg(
        mean_runtime_sec=("runtime_sec", "mean"),
        median_runtime_sec=("runtime_sec", "median"),
        mean_matches=("num_matches", "mean"),
        all_source_found=("source_found", "all"),
    )
)
summary_path = OUTPUT_DIR / "ncbi_full_genome_experiment_summary.csv"
summary.to_csv(summary_path, index=False)
print("Saved:", summary_path)
summary

## 5. Trực quan hóa full genome

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

def save_current_figure(filename: str):
    path = OUTPUT_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    print("Saved:", path)

if completed.empty:
    print("Không có lượt chạy hoàn thành để vẽ biểu đồ runtime.")
else:
    plot_df = (
        completed.groupby(["algorithm_key", "k"], as_index=False)
        .agg(mean_runtime_sec=("runtime_sec", "mean"))
    )
    pivot_df = plot_df.pivot(index="algorithm_key", columns="k", values="mean_runtime_sec")
    ax = pivot_df.plot(kind="bar", figsize=(10, 5), width=0.82)
    ax.set_title("Full genome runtime by algorithm and k")
    ax.set_xlabel("Algorithm")
    ax.set_ylabel("Mean runtime (sec)")
    ax.set_yscale("log")
    ax.legend(title="k")
    save_current_figure("full_genome_runtime_by_algorithm_k.png")
    plt.show()

In [ ]:
if completed.empty:
    print("Không có lượt chạy hoàn thành để vẽ biểu đồ theo pattern length.")
else:
    pattern_plot_df = (
        completed.groupby(["algorithm_key", "pattern_length"], as_index=False)
        .agg(mean_runtime_sec=("runtime_sec", "mean"))
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    for algorithm_key, group in pattern_plot_df.groupby("algorithm_key"):
        ax.plot(group["pattern_length"], group["mean_runtime_sec"], marker="o", label=algorithm_key)
    ax.set_title("Full genome runtime by pattern length")
    ax.set_xlabel("Pattern length (bp)")
    ax.set_ylabel("Mean runtime (sec, log scale)")
    ax.set_yscale("log")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
    save_current_figure("full_genome_runtime_by_pattern_length.png")
    plt.show()

In [ ]:
status_plot_df = status_summary.pivot(index="algorithm_key", columns="status", values="runs").fillna(0)
ax = status_plot_df.plot(kind="bar", stacked=True, figsize=(10, 5))
ax.set_title("Full genome run status by algorithm")
ax.set_xlabel("Algorithm")
ax.set_ylabel("Number of runs")
ax.legend(title="Status")
save_current_figure("full_genome_run_status.png")
plt.show()

In [ ]:
if completed.empty:
    print("Chưa có thuật toán nào hoàn thành trong giới hạn timeout.")
else:
    overall_rank = (
        completed.groupby("algorithm_key", as_index=False)
        .agg(mean_runtime_sec=("runtime_sec", "mean"), median_runtime_sec=("runtime_sec", "median"))
        .sort_values("mean_runtime_sec")
    )
    fastest = overall_rank.iloc[0]
    print(f"Thuật toán nhanh nhất trung bình trên full genome: {fastest.algorithm_key} ({fastest.mean_runtime_sec:.6f}s).")
    display(overall_rank)

if not accuracy_summary.empty and accuracy_summary["source_found_rate"].min() < 1.0:
    print("Có thuật toán/case hoàn thành nhưng không tìm thấy vị trí pattern nguồn:")
    display(accuracy_summary[accuracy_summary["source_found_rate"] < 1.0])
elif not accuracy_summary.empty:
    print("Các lượt chạy hoàn thành đều tìm thấy vị trí pattern nguồn.")